# 15 · Quality & Accuracy Metrics
**Evaluate classification quality against ground-truth labels from `Scopus_Content_Collapsed.xlsx`.**

| Step | What happens |
|------|--------------|
| 1 | Load a completed run and align it with ground-truth |
| 2 | Overall accuracy: contact_reason L1 and L2 |
| 3 | Effort score accuracy vs `Csat Effort Answer Desc` |
| 4 | Confusion matrix |
| 5 | Precision / recall / F1 per category |
| 6 | Confidence calibration (confidence score vs. actual correctness) |
| 7 | Error analysis — worst-performing categories |

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── Settings ─────────────────────────────────────────────────────────────
# Use the enhanced run (notebook 12) — it has confidence scores.
# Switch to tour_sample_enriched.xlsx (notebook 10) if you only need accuracy.
RUN_FILE       = "runs/output/enhanced_sample_enriched.xlsx"
DATA_FILE      = "data/Scopus_Content_Collapsed.xlsx"

ID_COL         = "Reference #"

# Contact reason
PRED_L1        = "contact_reason_primary_label"
PRED_L2        = "contact_reason_primary_code"
GT_L1          = "Contact_Reason_Level_1"
GT_L2          = "Contact_Reason_Level_2"

# Effort
PRED_EFFORT    = "effort_primary_label"
GT_EFFORT      = "Csat Effort Answer Desc"

# Confidence — enrich_with_confidence writes '_confidence' (not a per-module column)
CONF_COL       = "_confidence"

## Step 1 · Load and align

In [ ]:
run_path = Path(RUN_FILE)
if not run_path.exists():
    raise FileNotFoundError(
        f"{RUN_FILE} not found. Run notebook 10 (Pipeline Tour) first to generate it."
    )

df = pd.read_excel(run_path)
df[ID_COL] = df[ID_COL].astype(str)
print(f"Run file: {len(df)} rows × {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Confirm which GT columns are present in the run file
available_gt = [c for c in [GT_L1, GT_L2, GT_EFFORT] if c in df.columns]
print(f"Ground-truth columns available: {available_gt}")

available_pred = [c for c in [PRED_L1, PRED_L2, PRED_EFFORT] if c in df.columns]
print(f"Prediction columns available  : {available_pred}")

conf_available = CONF_COL in df.columns
print(f"Confidence column available   : {conf_available} ({CONF_COL})")

## Step 2 · Contact reason accuracy

In [ ]:
def accuracy_report(df, pred_col, gt_col, name=""):
    if pred_col not in df.columns or gt_col not in df.columns:
        print(f"  {name}: columns missing ({pred_col}, {gt_col})")
        return None
    valid = df[[pred_col, gt_col]].dropna()
    if len(valid) == 0:
        print(f"  {name}: no valid rows")
        return None
    n_correct = (valid[pred_col] == valid[gt_col]).sum()
    acc = n_correct / len(valid)
    print(f"  {name:<30}  {n_correct}/{len(valid)}  ({acc:.1%})")
    return valid

print("=== Contact Reason Accuracy ===")
valid_l1 = accuracy_report(df, PRED_L1, GT_L1, "L1 label exact match")
valid_l2 = accuracy_report(df, PRED_L2, GT_L2, "L2 code exact match")

## Step 3 · Effort accuracy

In [ ]:
print("=== Effort Accuracy ===")

# Normalise effort labels for comparison
effort_norm = {
    "low effort": "Low effort", "low": "Low effort",
    "high effort": "High effort", "high": "High effort",
    "medium effort": "Medium effort", "medium": "Medium effort",
}

if PRED_EFFORT in df.columns and GT_EFFORT in df.columns:
    df_eff = df[[PRED_EFFORT, GT_EFFORT]].dropna().copy()
    df_eff["pred_norm"] = df_eff[PRED_EFFORT].str.lower().map(
        lambda x: effort_norm.get(str(x).lower(), str(x)))
    df_eff["gt_norm"]   = df_eff[GT_EFFORT].str.strip()
    n_correct = (df_eff["pred_norm"] == df_eff["gt_norm"]).sum()
    print(f"  Effort (normalised)  {n_correct}/{len(df_eff)}  ({n_correct/len(df_eff):.1%})")
    print()
    print("Predicted distribution:")
    print(df_eff["pred_norm"].value_counts().to_string())
    print()
    print("Ground-truth distribution:")
    print(df_eff["gt_norm"].value_counts().to_string())
elif PRED_EFFORT not in df.columns:
    # Try effort_score instead
    effort_score_col = "effort_score"
    if effort_score_col in df.columns:
        print(f"  Using {effort_score_col} column instead of {PRED_EFFORT}")
        print(df[effort_score_col].describe().round(2).to_string())
    else:
        print(f"  Effort columns not found: {PRED_EFFORT}, {effort_score_col}")
else:
    print(f"  GT column {GT_EFFORT} not in output.")

## Step 4 · Confusion matrix

In [ ]:
if valid_l1 is not None and len(valid_l1) > 0:
    confusion = pd.crosstab(
        valid_l1[GT_L1].rename("Ground Truth"),
        valid_l1[PRED_L1].rename("Predicted"),
    )
    print(f"Confusion matrix ({len(confusion)} GT categories × {len(confusion.columns)} predicted):")
    display(confusion)

    # Heatmap
    if len(confusion) <= 20:
        fig, ax = plt.subplots(figsize=(max(8, len(confusion.columns) * 0.8),
                                        max(6, len(confusion) * 0.6)))
        im = ax.imshow(confusion.values, aspect="auto", cmap="Blues")
        ax.set_xticks(range(len(confusion.columns)))
        ax.set_xticklabels(confusion.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(confusion.index)))
        ax.set_yticklabels(confusion.index, fontsize=7)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Ground Truth")
        ax.set_title("Contact Reason L1 Confusion Matrix")
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()
    else:
        print("(Too many categories for heatmap — see table above)")
else:
    print("No valid L1 data for confusion matrix.")

## Step 5 · Precision, Recall, F1 per category

In [ ]:
if valid_l1 is not None and len(valid_l1) > 0:
    try:
        from sklearn.metrics import classification_report
        report = classification_report(
            valid_l1[GT_L1], valid_l1[PRED_L1],
            output_dict=True, zero_division=0
        )
        df_report = pd.DataFrame(report).T
        df_report = df_report.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
        df_report = df_report.round(3).sort_values("f1-score", ascending=False)
        print("Per-category classification report:")
        display(df_report)
    except ImportError:
        print("sklearn not installed. Computing manually...")

        categories = valid_l1[GT_L1].unique()
        rows = []
        for cat in sorted(categories):
            tp = ((valid_l1[PRED_L1] == cat) & (valid_l1[GT_L1] == cat)).sum()
            fp = ((valid_l1[PRED_L1] == cat) & (valid_l1[GT_L1] != cat)).sum()
            fn = ((valid_l1[PRED_L1] != cat) & (valid_l1[GT_L1] == cat)).sum()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            rows.append({"category": cat, "precision": round(precision, 3),
                         "recall": round(recall, 3), "f1": round(f1, 3),
                         "support": int(tp + fn)})
        df_report = pd.DataFrame(rows).set_index("category").sort_values("f1", ascending=False)
        display(df_report)
else:
    print("No valid L1 data for precision/recall analysis.")

## Step 6 · Confidence calibration
If confidence scores are available, check whether high-confidence predictions are more accurate.

In [ ]:
if conf_available and PRED_L1 in df.columns and GT_L1 in df.columns:
    df_cal = df[[CONF_COL, PRED_L1, GT_L1]].dropna()
    df_cal["correct"] = (df_cal[PRED_L1] == df_cal[GT_L1]).astype(int)

    # Bin into confidence deciles
    df_cal["conf_bin"] = pd.cut(df_cal[CONF_COL], bins=10, labels=False)
    calibration = df_cal.groupby("conf_bin").agg(
        mean_confidence=(CONF_COL, "mean"),
        accuracy=("correct", "mean"),
        count=("correct", "count"),
    ).reset_index()

    print("Confidence calibration (10 bins):")
    print(calibration.round(3).to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
    ax.scatter(calibration["mean_confidence"], calibration["accuracy"],
               s=calibration["count"] * 5, color="steelblue", zorder=3)
    ax.plot(calibration["mean_confidence"], calibration["accuracy"],
            "o-", color="steelblue", label="Actual")
    ax.set_xlabel("Mean confidence score")
    ax.set_ylabel("Accuracy")
    ax.set_title("Confidence Calibration Plot")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
else:
    if not conf_available:
        print(f"Confidence column '{CONF_COL}' not in output.")
        print("Enable ENABLE_CONFIDENCE=True in notebook 12 to generate it.")
    else:
        print("Prediction or GT columns missing for calibration.")

## Step 7 · Error analysis — worst-performing categories

In [ ]:
if valid_l1 is not None and len(valid_l1) > 0:
    errors = valid_l1[valid_l1[PRED_L1] != valid_l1[GT_L1]].copy()
    print(f"Total errors: {len(errors)}/{len(valid_l1)} ({len(errors)/len(valid_l1):.1%} error rate)")
    print()

    # Most common error patterns
    error_pairs = (
        errors.groupby([GT_L1, PRED_L1])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
    )
    print("Top error patterns (ground truth → predicted):")
    display(error_pairs.head(10))
else:
    print("No valid L1 data for error analysis.")

In [ ]:
# Categories with highest error rate (min 3 support)
if valid_l1 is not None and len(valid_l1) > 0:
    stats = valid_l1.copy()
    stats["correct"] = (stats[PRED_L1] == stats[GT_L1])
    cat_stats = (
        stats.groupby(GT_L1)
             .agg(total=("correct", "count"), correct=("correct", "sum"))
             .assign(error_rate=lambda d: 1 - d["correct"] / d["total"])
             .query("total >= 3")
             .sort_values("error_rate", ascending=False)
             .round(3)
    )
    print("Categories with highest error rate (min 3 examples):")
    display(cat_stats.head(10))

    if len(cat_stats) > 0:
        worst = cat_stats.head(8)
        fig, ax = plt.subplots(figsize=(8, max(3, len(worst) * 0.5)))
        ax.barh(worst.index, worst["error_rate"] * 100, color="tomato")
        ax.set_xlabel("Error rate (%)")
        ax.set_title("Top error-prone categories")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter())
        plt.tight_layout()
        plt.show()

In [ ]:
# Example errors — inspect the conversation text for a specific category
INSPECT_CATEGORY = None   # Set to a GT_L1 value to inspect, or leave None for worst

if valid_l1 is not None and len(errors) > 0:
    if INSPECT_CATEGORY is None and len(cat_stats) > 0:
        INSPECT_CATEGORY = cat_stats.index[0]

    if INSPECT_CATEGORY:
        cat_errors = df[
            (df[GT_L1] == INSPECT_CATEGORY) & (df[PRED_L1] != INSPECT_CATEGORY)
        ][[ID_COL, "Conversation" if "Conversation" in df.columns else df.columns[1],
           GT_L1, PRED_L1]].head(3)
        print(f"Error examples for '{INSPECT_CATEGORY}':")
        for _, row in cat_errors.iterrows():
            print(f"\n  [{row[ID_COL]}]")
            print(f"  GT: {row[GT_L1]}")
            print(f"  Pred: {row[PRED_L1]}")
            text_col = "Conversation" if "Conversation" in df.columns else df.columns[1]
            print(f"  Text: {str(row[text_col])[:300]}...")